# 🎬 Supernan Hindi Dubbing Pipeline

**Zero-cost pipeline: English video → Hindi dubbed video with lip sync**

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

Pipeline stages:
1. Install dependencies
2. Clone VideoReTalking & GFPGAN
3. Upload & extract 15-second clip
4. Whisper transcription
5. IndicTrans2 Hindi translation
6. Coqui XTTS v2 voice cloning
7. Audio speed adjustment
8. VideoReTalking lip-sync
9. GFPGAN face enhancement
10. Download output

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install System Deps ───────────────────────────────────────────────
!apt-get install -y ffmpeg wget unzip -q
print('✓ ffmpeg installed')

In [ ]:
# ── Cell 3: Install Python packages ──────────────────────────────────────────
# PyTorch with CUDA (Colab already has this, but ensure correct version)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# Core pipeline deps
!pip install -q openai-whisper
!pip install -q TTS
!pip install -q transformers sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransTokenizer.git
!pip install -q pydub librosa soundfile deep-translator
!pip install -q basicsr facexlib gfpgan realesrgan
print('✓ All packages installed')

In [ ]:
# ── Cell 4: Clone Repos ───────────────────────────────────────────────────────
import os

# VideoReTalking
if not os.path.exists('VideoReTalking'):
    !git clone -q https://github.com/vinthony/video-retalking VideoReTalking
    !pip install -q -r VideoReTalking/requirements.txt
print('✓ VideoReTalking ready')

# GFPGAN
if not os.path.exists('GFPGAN'):
    !git clone -q https://github.com/TencentARC/GFPGAN GFPGAN
    !pip install -q basicsr facexlib gfpgan
    %cd GFPGAN
    !python setup.py develop -q
    %cd ..
print('✓ GFPGAN ready')

In [ ]:
# ── Cell 5: Download VideoReTalking Weights ───────────────────────────────────
os.makedirs('models/VideoReTalking', exist_ok=True)

CHECKPOINTS = {
    '30_net_gen.pth':   'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/30_net_gen.pth',
    'DNet.pt':          'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/DNet.pt',
    'ENet.pth':         'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/ENet.pth',
    'GFPGANv1.3.pth':   'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/GFPGANv1.3.pth',
    'GPEN-BFR-512.pth': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/GPEN-BFR-512.pth',
    'ParseNet-latest.pth': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/ParseNet-latest.pth',
    'RetinaFace-R50.pth':  'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/RetinaFace-R50.pth',
    'shape_predictor_68_face_landmarks.dat': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/shape_predictor_68_face_landmarks.dat',
    'BFM.zip': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/BFM.zip',
}

for fname, url in CHECKPOINTS.items():
    dest = f'models/VideoReTalking/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        !wget -q --show-progress -O {dest} {url}
    else:
        print(f'✓ {fname} already downloaded')

# Unzip BFM
if not os.path.isdir('models/VideoReTalking/BFM'):
    !unzip -q models/VideoReTalking/BFM.zip -d models/VideoReTalking/

print('✓ All VideoReTalking weights ready')

In [ ]:
# ── Cell 6: Download GFPGAN Weights ──────────────────────────────────────────
os.makedirs('GFPGAN/experiments/pretrained_models', exist_ok=True)
gfpgan_weight = 'GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth'
if not os.path.exists(gfpgan_weight):
    !wget -q --show-progress -O {gfpgan_weight} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
print('✓ GFPGAN weights ready')

In [ ]:
# ── Cell 7: Upload Input Video ────────────────────────────────────────────────
from google.colab import files

print('Upload your source video (the Supernan training video)')
uploaded = files.upload()
INPUT_VIDEO = list(uploaded.keys())[0]
print(f'✓ Uploaded: {INPUT_VIDEO}')

In [ ]:
# ── Cell 8: Configuration ────────────────────────────────────────────────────
# ⬇️ Change these if you want a different segment
START_SEC = 15
END_SEC   = 30

os.makedirs('workspace', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print(f'Processing segment: {START_SEC}s – {END_SEC}s ({END_SEC - START_SEC}s clip)')

In [ ]:
# ── Cell 9: Stage 1 — Extract Clip ───────────────────────────────────────────
CLIP = 'workspace/clip.mp4'
duration = END_SEC - START_SEC

!ffmpeg -y -ss {START_SEC} -i "{INPUT_VIDEO}" -t {duration} \
    -c:v libx264 -c:a aac -preset fast -crf 18 {CLIP} -loglevel warning

from IPython.display import Video
print('✓ Clip extracted')
Video(CLIP, width=640)

In [ ]:
# ── Cell 10: Stage 2 — Extract Audio ─────────────────────────────────────────
AUDIO_16K = 'workspace/clip_audio_16k.wav'
AUDIO_REF = 'workspace/clip_audio_ref44k.wav'

!ffmpeg -y -i {CLIP} -vn -acodec pcm_s16le -ar 16000 -ac 1 {AUDIO_16K} -loglevel warning
!ffmpeg -y -i {CLIP} -vn -acodec pcm_s16le -ar 44100 -ac 1 {AUDIO_REF} -loglevel warning
print(f'✓ Audio extracted: 16kHz (Whisper) + 44kHz (XTTS reference)')

In [ ]:
# ── Cell 11: Stage 3 — Transcribe with Whisper ───────────────────────────────
import whisper

print('Loading Whisper medium model (downloading ~1.5 GB on first run)...')
model = whisper.load_model('medium')  # 'small' is faster; 'large' is more accurate

result = model.transcribe(AUDIO_16K, language='en', word_timestamps=True, verbose=False)
ENGLISH_TEXT = result['text'].strip()

with open('workspace/transcript_en.txt', 'w') as f:
    f.write(ENGLISH_TEXT)

print(f'✓ Transcript:\n{ENGLISH_TEXT}')

In [ ]:
# ── Cell 12: Stage 4 — Translate to Hindi (IndicTrans2) ──────────────────────
def translate_indictrans2(text):
    """Primary: IndicTrans2 (context-aware, much better than Google Translate)"""
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        from IndicTransTokenizer import IndicProcessor
        import torch

        MODEL = 'ai4bharat/indictrans2-en-indic-1B'
        print('Loading IndicTrans2 (downloading ~2 GB on first run)...')
        tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True).cuda()
        ip = IndicProcessor(inference=True)

        sentences = [s.strip() for s in text.split('.') if s.strip()]
        batch = ip.preprocess_batch(sentences, src_lang='eng_Latn', tgt_lang='hin_Deva')
        inputs = tokenizer(batch, truncation=True, padding='longest', return_tensors='pt').to('cuda')
        with torch.no_grad():
            outputs = model.generate(**inputs, num_beams=5, max_length=256)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang='hin_Deva')
        return ' '.join(postprocessed)
    except Exception as e:
        print(f'IndicTrans2 failed: {e}. Falling back to Google Translate...')
        return None

HINDI_TEXT = translate_indictrans2(ENGLISH_TEXT)

if HINDI_TEXT is None:
    from deep_translator import GoogleTranslator
    HINDI_TEXT = GoogleTranslator(source='en', target='hi').translate(ENGLISH_TEXT)
    print('Used fallback: Google Translate')

with open('workspace/translation_hi.txt', 'w', encoding='utf-8') as f:
    f.write(HINDI_TEXT)

print(f'✓ Hindi translation:\n{HINDI_TEXT}')

In [ ]:
# ── Cell 13: Stage 5 — Coqui XTTS v2 Voice Cloning ──────────────────────────
from TTS.api import TTS
import re

TTS_RAW = 'workspace/tts_raw.wav'

print('Loading Coqui XTTS v2 (downloading ~1.8 GB on first run)...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').cuda()

# Split into chunks for stability (XTTS works best on <300 char segments)
def chunk_text(text, max_chars=250):
    sentences = re.split(r'(?<=[।|.!?])\s+', text)
    chunks, current = [], ''
    for s in sentences:
        if len(current) + len(s) + 1 <= max_chars:
            current = (current + ' ' + s).strip()
        else:
            if current: chunks.append(current)
            current = s
    if current: chunks.append(current)
    return chunks or [text]

chunks = chunk_text(HINDI_TEXT)
print(f'Synthesizing {len(chunks)} chunk(s)...')

if len(chunks) == 1:
    tts.tts_to_file(text=HINDI_TEXT, speaker_wav=AUDIO_REF, language='hi', file_path=TTS_RAW)
else:
    import os
    os.makedirs('workspace/tts_chunks', exist_ok=True)
    chunk_paths = []
    for i, chunk in enumerate(chunks):
        p = f'workspace/tts_chunks/tts_{i:04d}.wav'
        print(f'  Chunk {i+1}/{len(chunks)}: {chunk[:50]}...')
        tts.tts_to_file(text=chunk, speaker_wav=AUDIO_REF, language='hi', file_path=p)
        chunk_paths.append(p)
    
    # Concatenate chunks
    concat_list = 'workspace/concat.txt'
    with open(concat_list, 'w') as f:
        for p in chunk_paths:
            f.write(f"file '{os.path.abspath(p)}'\n")
    !ffmpeg -y -f concat -safe 0 -i {concat_list} -c copy {TTS_RAW} -loglevel warning

print(f'✓ Hindi TTS audio: {TTS_RAW}')

In [ ]:
# ── Cell 14: Stage 6 — Adjust Audio Speed ────────────────────────────────────
import subprocess

TTS_ADJ = 'workspace/tts_adjusted.wav'

def get_duration(path):
    r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',path],
                       capture_output=True, text=True)
    return float(r.stdout.strip())

def build_atempo(ratio):
    filters = []
    r = ratio
    while r < 0.5:
        filters.append('atempo=0.5'); r /= 0.5
    while r > 2.0:
        filters.append('atempo=2.0'); r /= 2.0
    filters.append(f'atempo={r:.6f}')
    return ','.join(filters)

tts_dur = get_duration(TTS_RAW)
clip_dur = get_duration(CLIP)
ratio = tts_dur / clip_dur
atempo = build_atempo(ratio)

print(f'TTS duration: {tts_dur:.2f}s | Clip duration: {clip_dur:.2f}s | ratio={ratio:.3f}')
!ffmpeg -y -i {TTS_RAW} -filter:a "{atempo}" -ar 44100 {TTS_ADJ} -loglevel warning
print(f'✓ Audio adjusted to {clip_dur:.2f}s')

In [ ]:
# ── Cell 15: Stage 7 — VideoReTalking Lip Sync ───────────────────────────────
import sys

LIPSYNC = 'workspace/lipsync.mp4'
CHECKPOINT_DIR = 'models/VideoReTalking'

print('Running VideoReTalking (this takes ~10-20 min on free T4)...')

result = subprocess.run([
    sys.executable, 'VideoReTalking/inference.py',
    '--face', CLIP,
    '--audio', TTS_ADJ,
    '--outfile', LIPSYNC,
    '--checkpoint_dir', CHECKPOINT_DIR,
], cwd='.')

if result.returncode != 0:
    print('⚠️ VideoReTalking failed. Falling back to simple audio mux...')
    !ffmpeg -y -i {CLIP} -i {TTS_ADJ} -map 0:v -map 1:a -c:v copy -c:a aac -shortest {LIPSYNC} -loglevel warning

print(f'✓ Lip-sync complete: {LIPSYNC}')
Video(LIPSYNC, width=640)

In [ ]:
# ── Cell 16: Stage 8 — GFPGAN Face Enhancement ───────────────────────────────
FINAL = 'output/final_dubbed.mp4'
FRAMES_DIR = 'workspace/frames_raw'
ENHANCED_DIR = 'workspace/gfpgan_out'

os.makedirs(FRAMES_DIR, exist_ok=True)

# Extract frames
print('Extracting frames...')
!ffmpeg -y -i {LIPSYNC} -qscale:v 1 -qmin 1 {FRAMES_DIR}/frame_%06d.png -loglevel warning

# Run GFPGAN
print('Running GFPGAN face restoration...')
!python GFPGAN/inference_gfpgan.py \
    -i {FRAMES_DIR} \
    -o {ENHANCED_DIR} \
    -v 1.4 -s 1 \
    --bg_upsampler None

# Get FPS
fps_r = subprocess.run(['ffprobe','-v','quiet','-select_streams','v:0',
                        '-show_entries','stream=r_frame_rate','-of','csv=p=0', LIPSYNC],
                       capture_output=True, text=True)
FPS = fps_r.stdout.strip()

# Re-encode with enhanced frames
RESTORED = f'{ENHANCED_DIR}/restored_imgs'
print('Re-encoding final video...')
!ffmpeg -y \
    -framerate {FPS} -pattern_type glob -i "{RESTORED}/*.png" \
    -i {LIPSYNC} \
    -map 0:v -map 1:a \
    -c:v libx264 -crf 17 -preset slow \
    -c:a aac -shortest \
    {FINAL} -loglevel warning

final_dur = get_duration(FINAL)
print(f'✓ Final output: {FINAL} ({final_dur:.2f}s)')
Video(FINAL, width=640)

In [ ]:
# ── Cell 17: Download Output ──────────────────────────────────────────────────
from google.colab import files
files.download(FINAL)
print(f'✓ Download started for {FINAL}')

---
## 📊 Pipeline Summary

| Stage | Tool | Duration |
|---|---|---|
| Clip Extract | ffmpeg | ~5s |
| Audio Extract | ffmpeg | ~2s |
| Transcription | Whisper medium | ~30s |
| Translation | IndicTrans2 | ~60s |
| Voice Clone | Coqui XTTS v2 | ~3-5 min |
| Speed Adjust | ffmpeg atempo | ~5s |
| Lip Sync | VideoReTalking | ~10-20 min |
| Face Enhance | GFPGAN v1.4 | ~2-3 min |

**Total on free Colab T4: ~15-30 minutes for a 15-second clip**

**Cost: ₹0** ✅